In [4]:
import torch
from torch import nn

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()

embedding = nn.Embedding(200021, 128, device=DEVICE)

@triton.jit
def _attention_bwd_pre_process(o_ptr, do_ptr, delta_ptr,
                               n_ctx: tl.constexpr,
                               pre_block: tl.constexpr,
                               heads: tl.constexpr,
                               hidden: tl.constexpr):
  pre_block_per_nctx = tl.program_id(0)
  off_h = tl.program_id(1)
  # for off_h in range(heads):
  #   for pre_block_per_nctx in range(n_ctx//pre_block):
  offs_pre_block = pre_block_per_nctx*pre_block + tl.arange(0, pre_block)
  offs_hid = tl.arange(0, hidden)
  offset = off_h*n_ctx*hidden + offs_pre_block[:,None]*hidden + offs_hid[None,:]
  # print(f"O and do offset({off_h}, {pre_block_per_nctx}) : {offset}")
  # o = torch.randn_like(offset, dtype=tl.bfloat16)
  # do = torch.rand_like(offset, dtype=tl.bfloat16)
  # o_od = torch.sum(o*do, 1)
  # print(f"O_do result offset({off_h}, {pre_block_per_nctx}) : {o_od.shape}")
  o = tl.load(o_ptr + offset)
  do = tl.load(do_ptr + offset)
  o_do = tl.sum(o*do, axis=1)
  # print(f"O_do result offset({off_h}, {pre_block_per_nctx}) : {off_h*n_ctx + offs_pre_block}")
  delta = delta_ptr + off_h*n_ctx + offs_pre_block
  tl.store(delta, o_do)

num_heads = 8
embedded_tensor = embedding(torch.randint(low=0, high=200021, size=(512*4,), device=DEVICE))
input_q = embedded_tensor.unsqueeze(0).expand(num_heads, -1, -1)
input_k = embedded_tensor.unsqueeze(0).expand(num_heads, -1, -1)
input_v = embedded_tensor.unsqueeze(0).expand(num_heads, -1, -1)
o = torch.rand_like(input_q)
do = torch.rand_like(input_q)
BLOCK_M = 64
BLOCK_N = 32
pre_block = 128
num_hiddens = input_q.shape[-1]
n_ctx = input_q.shape[1]
grid = (input_q.shape[1]//pre_block, num_heads, 1)
print(f"Grid : {grid}, q: {input_q.shape}, k: {input_k.shape}, v: {input_v.shape} ")
delta = torch.empty((input_q.shape[0], input_q.shape[1]), device=input_q.device, dtype=torch.float32)
# Preprocess
_attention_bwd_pre_process[grid](o, do, delta, n_ctx, pre_block, num_heads, num_hiddens)
print(delta)


Grid : (16, 8, 1), q: torch.Size([8, 2048, 128]), k: torch.Size([8, 2048, 128]), v: torch.Size([8, 2048, 128]) 
tensor([[31.9392, 27.0023, 31.7695,  ..., 29.5097, 32.2574, 33.2953],
        [29.3994, 35.2510, 31.1757,  ..., 30.6001, 31.2799, 33.7668],
        [32.6903, 30.7698, 31.0949,  ..., 30.5769, 31.8751, 36.7761],
        ...,
        [37.0844, 32.4867, 34.3689,  ..., 31.0852, 34.3999, 32.5404],
        [30.3746, 33.5656, 32.4827,  ..., 32.8728, 32.1286, 33.7698],
        [32.5119, 34.5369, 29.9424,  ..., 35.0240, 31.4271, 29.9496]],
       device='cuda:0')
